# CAP 4611 – Algorithms for Machine Learning · Spring 2026

## Assignment 3 - A3 Classification

## Section 1 - Load Data and Perform Basic EDA (7 pts)

### Section 1.1 - Import Libraries (1 pt)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import missingno as msno
from scipy import stats
import sklearn

print("✓ All libraries imported successfully!")
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"sklearn : {sklearn.__version__}")

### Section 1.2 - Import the Dataset and Show Row/Columns Count (1 pt)

In [ ]:
# .shape[0] returns the number of rows, .shape[1] returns the number of columns

df = pd.read_csv('hrdata2.csv')

print(f"Number of Rows      : {df.shape[0]}")
print(f"Number of Columns   : {df.shape[1]}")

### Section 1.3 - First 5 and Last 5 Rows

In [ ]:
# head() displays the first 5 rows by default
# lets us confirm the data loaded correctly and see all column names
print("=== Top 5 Rows ===")
df.head()

In [ ]:
# tail() displays the last 5 rows by default
# helps us check for any empty or corrupted rows at the end of the file
print("=== Last 5 Rows ===")
df.tail()

### Section 1.4 - Show How Many Columns Contain Null Values (1 pt)

In [ ]:
# isnull() returns True/False for every cell in the dataframe
# sum() counts all the Trues per column giving us the missing count per feature
# this tells us exactly which columns need attention before modeling

null_counts = df.isnull().sum()
print("=== Null Values Per Column ===")
print(null_counts)
print(f"\nTotal columns WITH null values : {(null_counts > 0).sum()}")
print(f"Total missing values in dataset. : {null_counts.sum()}")

### Section 1.5 - Plot the Target Distribution (3 pts)
- Plot the target distribution. Discuss class imbalance, potential issues, and possible solutions

In [ ]:
# Count how many records belong to each target class
# target 0 = candidate NOT looking for a job change
# target 1 = candidate IS looking for a job change

target_counts = df['target'].value_counts()
labels = ['Not Looking (0)', 'Looking (1)']
total = len(df)
plt.figure(figsize=(7, 5))
bars = plt.bar(labels, target_counts.values, color=['steelblue', 'tomato'], edgecolor='white', linewidth=0.5)

# Add the exactly count on top of each bar so its easy to read
for bar, count in zip(bars, target_counts.values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30, str(count), ha='center', va='bottom', fontsize=12)

plt.xlabel('Target Class')
plt.ylabel('Number of Records')
plt.title('Target Variable Distribution')
plt.tight_layout()
plt.show()

# Print the exact percentage breakdown of each class
for label, count in zip(['0 - Not Looking', '1 - Looking'], target_counts.values):
    print(f"Class {label}: {count} records ({count/total*100:.1f}%)")

### Section 1 : Observation Cell
- Class Imbalance: The dataset is imbalanced. Class 0 (not looking for a job change) makes up roughly 75% of the data while Class 1 (looking for a job change) makes up only about 25%
- Potential Issues: A model trained on this imbalanced data will be biased toward always predicting Class 0 since that is the majority. It can still achieve high overall accuracy by doing this, but it will miss most of the Class 1 candidates - which defeats the entire purpose of building the model.
- Possible Solution:
    - SMOTE - Creates synthetic samples of the majority class to balance the training set (we apply this in Section 3)
    - Undersampling - randomly remove records from the majority class
    - class_weight='Balanced' - tells the model to penalize mistakes on the minority class more heavily
    - Use better metrics like AUC-ROC, F1-score, and recall instead of plain accuracy which is misleading on imbalanced data

## Section 2 - Feature Selection and Pre-processing (32 pts)

### Section 2.1 - Preprocessing City (4 pts)

#### Section 2.1.1 - Plot Number of Records per City in Descending Order (1 pt)

In [ ]:
# Count the number of records per city and sort in descending order
# This helps us visually identify which cities have the most candidates
# We use this plot to decide which top 4 cities to keep in section 2.1.2

city_counts = df['city'].value_counts()

plt.figure(figsize=(14, 5))
city_counts.plot(kind='bar', color='steelblue', edgecolor='white')
plt.xlabel('City')
plt.ylabel('Number of Records')
plt.title('Number of Records per City (Descending Order)')
plt.xticks(rotation=90, fontsize=6)
plt.tight_layout()
plt.show()

#### Section 2.1.2 - How Many Rows Belong to the Top 4 Cities vs Remaining (1 pt)

In [ ]:
# value_counts() already sorted the cities from highest to lowest
# head(4) grabs the top 4 cities by record count
# isin() checks which rows belong to those top 4 cities

top4_cities = city_counts.head(4).index.tolist()
print(f"Top 4 Cities: {top4_cities}")

top4_counts = df[df['city'].isin(top4_cities)].shape[0]
remaining_count = df[~df['city'].isin(top4_cities)].shape[0]

print(f"\nRows in top 4 cities      : {top4_counts}")
print(f"Rows in remaining cities    : {remaining_count}")

#### Section 2.1.3 - Replace Non-Top-4 Cities with city_others (1 pt)

In [ ]:
# Replace any city not in the top 4 with 'city_others'
# set() makes the lookup faster than a list
# np.where works like an if/else — if city is in top4, keep it, otherwise replace it

top4_set = set(top4_cities)
df['city'] = np.where(df['city'].isin(top4_set), df['city'], 'city_others')

print("City column updated.")
print("Unique values now:", df['city'].unique())

#### Section 2.1.4 - Show Sample Rows to Verify Changes (1 pt)

In [ ]:
# Display a random sample of rows to confirm the replacement worked correctly
# We should only see the 4 city names and city_others — nothing else

df[['enrollee_id', 'city']].sample(10, random_state=0)

### Section 2.2 - Education Level (4 pts)

#### Section 2.2.1 - Show Unique Values of Education Level (1 pt)

In [ ]:
# Show all unique values in education_level before we change anything
# We need to see the exact text values so we can map them correctly

print("Unique education_level values:")
print(df['education_level'].unique())


#### Section 2.2.2 - Convert Education Level to Ordinal Values (2 pts)

In [ ]:
# Map education levels to ordinal numbers based on academic rank
# Graduate = 0 (lowest), Masters = 1, Phd = 2 (highest)
# We use a dictionary + apply() + lambda to replace each value
# Ordinal encoding is used here because education level has a natural order

education_map = {
    'Graduate'  : 0,
    'Masters'   : 1,
    'Phd'       : 2
}

df['education_level'] = df['education_level'].apply(lambda x: education_map.get(x, np.nan))

print("Encoding Complete.")
print("Unique values after encoding:", sorted(df['education_level'].dropna().unique()))

#### Section 2.2.3 - Show Sample Data to Verify Changes (1 pt)

In [ ]:
# Sample rows to confirm education_level now contains 0, 1, or 2
# and no longer contains the original text values

df[['enrollee_id', 'education_level']].sample(10, random_state=0)

### Section 2.3 - Company Size (4 pts)

#### Section 2.3.1 - Show Unique Values of Company Size (1 pt)

In [ ]:
# Show all unique values before encoding
# company_size uses text ranges like '50-99' which we need to convert to numbers

print("Unique company_size values:")
print(df['company_size'].unique())

#### Section 2.3.2 - Convert Company Size to Ordinal Values (0 to 7)

In [ ]:
# Map company size ranges to ordinal integers 0 through 7
# The order reflects actual company size from smallest to largest
# This preserves the natural ordering — a larger number means a bigger company

company_size_map = {
    '<10'           : 0,
    '10/49'         : 1,
    '50-99'         : 2,
    '100-500'       : 3,
    '500-999'       : 4,
    '1000-4999'     : 5,
    '5000-9999'     : 6,
    '10000+'        : 7
}

df['company_size'] = df['company_size'].apply(lambda x: company_size_map.get(x, np.nan))
print("Encoding Complete.")



#### Section 2.3.3 - Show Updated Unique Values

In [ ]:
# Confirm the text values are now numbers
# dropna() excludes missing values, sorted() puts them in order

print("Unique company_size values after encoding:")
print(sorted(df['company_size'].dropna().unique()))

### Section 2.4 - Last New Job (4 pts)

#### Section 2.4.1 - Show Unique Values of Last New Job

In [ ]:
# Show all unique values in last_new_job before encoding
# This column represents how many years between the candidate's previous and current job

print("Unique last_new_job values:")
print(df['last_new_job'].unique())

#### Section 2.4.2 - Convert Last New Job to Numeric Values

In [ ]:
# Map last_new_job to numeric values
# 'never' means the candidate has never switched jobs before — maps to 0
# '>4' means more than 4 years gap — we cap it at 5 to keep it ordinal

last_new_job_map = {
    'never'     : 0,
    '1'         : 1,
    '2'         : 2,
    '3'         : 3,
    '4'         : 4,
    '>4'        : 5
}

df['last_new_job'] = df['last_new_job'].apply(lambda x: last_new_job_map.get(x, np.nan))
print("Encoding Complete.")

#### Section 2.4.3 - Show Updated Values

In [ ]:
# Confirm the values are now 0 through 5
# never=0, 1=1, 2=2, 3=3, 4=4, >4=5

print("Unique last_new_job values after encoding:")
print(sorted(df['last_new_job'].dropna().unique()))

### Section 2.5 - Other Columns (8 pts)

#### Section 2.5.1 - Show Unique Values of Remaining Categorical Columns (2 pts)

In [ ]:
# These columns contain text categories that need to be converted to numbers
# We loop through each one and print unique values so we know what to expect
# before applying get_dummies in the next step

cat_cols = ['company_type', 'major_discipline', 'enrolled_university', 'relevent_experience', 'gender', 'city']

for col in cat_cols:
    print(f"{col}: {df[col].unique()}")
    print()

#### Section 2.5.2 - Apply get_dummies to Categorical Columns (4 pts)

In [ ]:
# get_dummies creates a new binary column for each unique category value
# For example gender becomes gender_Male, gender_Female, gender_Other
# Each new column contains 1 if that was the value, 0 if not
# This is necessary because ML models cannot work with raw text values

df = pd.get_dummies(df, columns=cat_cols)

print("get_dummies encoding complete.")
print(f"New Shape: {df.shape}")

#### Section 2.5.3 - get_dummies vs OneHotEncoder (1 pt)

- Observations between get_dummies and OneHotencoder
    - get_dummies is a pandas function that converts categorical columns into binary columns directly in one step. It is simple and fast but does not learn from training data - if the test set has a category not seen during training, the columns will not match and the model will break
    - OneHotEncoder from sklearn is fitted on the training set first and then applied to the test set using the same learned categories. This prevents mismatched columns between train and test, making it safer and more reliable for production machine learning pipelines.

#### Section 2.5.4 - Show Top 5 and Last 5 Rows and Shape (1 pt)

In [ ]:
# Set pandas to show all columns since the table is now much wider
# after get_dummies we went from ~15 columns to many more binary columns

pd.set_option('display.max_columns', None)

print("=== Top 5 Rows ===")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("=== Last 5 Rows ===")
df.tail()

### Section 2.6 - Drop enrollee_id and Duplicate Columns (2 pts)

In [ ]:
# Drop enrollee_id since it is just a unique identifier with no predictive value
# Drop Unnamed: 0 which is a leftover index column from the original CSV file

cols_to_drop = ['enrollee_id']
if 'Unnamed: 0' in df.columns:
    cols_to_drop.append('Unnamed: 0')

df = df.drop(columns=cols_to_drop)

print(f"Dropped: {cols_to_drop}")
print(f"Shape after dropping: {df.shape}")

### Section 2.7 - Feature Scaling with MinMaxScaler (5 pts)

#### Section 2.7.1 - Apply MinMaxScaler to All Columns (2.5 pts)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# MinMaxScaler transforms every feature to a range between 0 and 1
# This prevents features with large ranges from dominating the model
# fit_transform learns the min and max from the data and applies the scaling

scaler = MinMaxScaler()
df_scaled_array = scaler.fit_transform(df)
df = pd.DataFrame(df_scaled_array, columns=df.columns)

print("Scaling complete.")
print(f"Shape: {df.shape}")

#### Section 2.7.2 - Show Sample Scaled Records (2.5 pts)

In [ ]:
# Confirm all values are now between 0 and 1
df.head()

#### Section 2.7.3 - Move Target Column to Last Position

In [ ]:
# Move the target column to the last position
# This is standard convention — features first, label last

target_col = df.pop('target')
df['target'] = target_col

print("Target moved to last column.")
print("Last 3 columns:", df.columns[-3:].tolist())

## Section 3 - X/Y Split, Stratified and SMOTE (15 pts)

### Section 3.1 - Copy Features into X and Target into Y (2 pts)

In [ ]:
# X contains all feature columns — everything we use to make predictions
# y contains only the target column — what we are trying to predict
# We separate them here so we can split into train/test sets in the next step

X = df.drop(columns=['target'])
y = df['target']

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")

### Section 3.2 - Show Class Ratio in Y (1.5 pts)

In [ ]:
# Show the percentage of class 0 vs class 1 in the full dataset
# This confirms the imbalance we saw in Section 1.5 is still present

print("Class ratio in Y:")
print(y.value_counts())
print()
print(y.value_counts(normalize=True).round(3))

### Section 3.3 - Train/Test Split (4 pts)

In [ ]:
from sklearn.model_selection import train_test_split

# Split data into 70% training and 30% test
# stratify=y ensures the same 83/17 class ratio is preserved in both splits
# random_state=0 makes the split reproducible so we get the same results every time

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=0, stratify=y
)

print(f"Training size : {X_train.shape[0]} rows")
print(f"Test size     : {X_test.shape[0]} rows")

### Section 3.4 - Show Class Ratio in y_train and y_test (1.5 pts)

In [ ]:
# Confirm stratification worked by checking class ratios in both splits
# Both should show roughly the same 83/17 ratio as the full dataset

print("Class ratio in y_train")
print(y_train.value_counts(normalize=True).round(3))

print("\nClass ratio in y_test")
print(y_test.value_counts(normalize=True).round(3))

### Section 3.5 - SMOTE (4 pts)

In [ ]:
from imblearn.over_sampling import SMOTE

# SMOTE = Synthetic Minority Oversampling Technique
# It creates brand new synthetic samples of the minority class (1.0)
# by looking at existing minority samples and generating similar ones
# We ONLY apply SMOTE to the training set — never the test set
# Applying it to the test set would be data leakage and give fake results

smote = SMOTE(random_state=0)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("SMOTE complete.")
print(f"Rebalanced training size:  {X_train_bal.shape[0]} rows")

### Section 3.6 - Show Class Ratio After SMOTE (2 pts)

In [ ]:
# Confirm both classes are now equally represented in the training set
# We should see 50% class 0 and 50% class 1
import pandas as pd
print("Class ratio in y_train after SMOTE:")
print(pd.Series(y_train_bal).value_counts())
print()
print(pd.Series(y_train_bal).value_counts(normalize=True).round(3))

## Section 4 - PCA and Logistic Regression (20 pts)

### Section 4.1 - PCA Pipeline to Find Best Number of Components (7 pts)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# We have 31 features — PCA reduces them to fewer dimensions
# while keeping as much useful information as possible
# We test every possible number of components from 1 to 31
# and measure accuracy using cross validation to find the best number

accuracies = []
n_features = X_train_bal.shape[1]

for n in range(1, n_features + 1):
    pipeline = Pipeline([
        ('pca', PCA(n_components=n)),
        ('lr', LogisticRegression(max_iter=1000, random_state=0))
    ])
    scores = cross_val_score(pipeline, X_train_bal, y_train_bal, cv=5, scoring='accuracy')
    accuracies.append(scores.mean())

# Plot accuracy vs number of PCA components
plt.figure(figsize=(10, 5))
plt.plot(range(1, n_features + 1), accuracies, marker='o', color='steelblue', markersize=4)
plt.xlabel('Number of PCA Components')
plt.ylabel('Cross-Validated Accuracy')
plt.title('PCA Component Count vs Logistic Regression Accuracy')
plt.xticks(range(1, n_features + 1))
plt.tight_layout()
plt.show()

best_n = accuracies.index(max(accuracies)) + 1
print(f"Best number of components: {best_n}")
print(f"Best Accuracy: {max(accuracies):.4f}")

### Section 4.2 - Evaluate Logistic Regression on Test Set Using Best PCA (4 pts)

In [ ]:
from sklearn.metrics import accuracy_score

# Train the final model using the best number of PCA components found above
# Then evaluate it on the test set using accuracy score

best_pipeline = Pipeline([
    ('pca', PCA(n_components=best_n)),
    ('lr', LogisticRegression(max_iter=1000, random_state=0))
])

best_pipeline.fit(X_train_bal, y_train_bal)
y_pred = best_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Test Set Accuracy with {best_n} PCA components: {accuracy:.4f}")

### Section 4.3 - Confusion Matrix (2 pts)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# The confusion matrix shows us 4 values:
# True Negatives (TN)  — correctly predicted NOT looking
# False Positives (FP) — predicted looking but actually NOT looking
# False Negatives (FN) — predicted NOT looking but actually looking
# True Positives (TP)  — correctly predicted looking

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not looking', 'Looking'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression with PCA')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives       (correctly predicted Not Looking)           : {tn}")
print(f"False Positives      (predicted Looking, Actually Not)           : {fp}")
print(f"False Negatives      (predicted Not looking, Actually Looking)   : {fn}")
print(f"True Positives       (correctly predicted Looking)               : {tp}")

### Section 4.4 - Precision, Recall, and F1 Score (2 pts)


In [ ]:
from sklearn.metrics import classification_report

# Precision — of all candidates we predicted as looking, how many actually were?
# Recall    — of all candidates actually looking, how many did we correctly catch?
# F1 Score  — harmonic mean of precision and recall, best for imbalanced data

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Looking', 'Looking']))

### Section 4.5 - ROC Curve and AUC (2 pts)


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# ROC curve shows the tradeoff between true positive rate and false positive rate
# AUC (Area Under Curve) — closer to 1.0 means a better model
# A random model would score 0.5, a perfect model scores 1.0

y_prob = best_pipeline.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC Curve (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression with PCA')
plt.legend()
plt.tight_layout()
plt.show()

print(f"AUC Score: {auc:.4f}")

### Section 4.6 - Precision-Recall Curve (3 pts)

In [ ]:
from sklearn.metrics import precision_recall_curve

# The precision-recall curve shows the tradeoff between precision and recall
# at different classification thresholds
# When threshold is high — model is very selective, high precision but low recall
# When threshold is low — model catches more positives, high recall but lower precision

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)

plt.figure(figsize=(8, 5))
plt.plot(recalls, precisions, color='tomato', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision Recall Curve - Logistic Regression with PCA')
plt.tight_layout()
plt.show()

### Section 4.7 - Observations
- When a recall is high the model catches more candidates who are looking for a job change but at the cost of more false positives. When precision is high the model is more selective but misses some true positives. The ideal threshold depends on whether the company cares more about not missing candidates (high recall) or not wasting resources on wrong predictions (high precision).

## Section 5 — Softmax Regression (3 pts)

### Section 5.1 - How Softmax Relates to Logistic Regression and Two Libraries (2 pts)

- Logistic Regression is used for binary classification - it predicts between two classes (0 and 1) using the sigmoid function which outputs a probability between 0 and 1.
- Softmax regression is the generalization of logistic regression to multiple classes. Instead of one sigmoid output, softmax produces a probability for each class and all probabilities sum to 1.0. When there are only 2 classes softmax and logistic regression produce identical results.
- Two libraries that can be used to build a softmax regression model:
    1. sklearn - LogisticRegression with multi_class='multinomial' and solver='lbfgs' handles softmax regression directly
    2. TensorfFlow/Keras - using a Dense Output layer with softmax as the activation function builds a softmax regression model as a neural network

### Section 5.2 - Weight and Bias Calculation for 4 Classes, 10 Features (1 pt)

- if we have 4 classes and 10 features, the softmax model has one of weights and one bias per class
    - weights: 4 classes x 10 features = 40 weights
    - Biases: 4 classes x 1 bias each = 4 biases
    - Total: 40 + 4 = 44 parameters

Each class gets its own weight vector of length 10 (one weight per feature) plus one bias term. The model computes a score for each class seperately and then softmax converts those 4 scores into probabilities.

## Section 6 - KNN (25 pts)

### Section 6.1 - KNN with Unbalanced Training Set (4 pts)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Train KNN with k=10 on the UNBALANCED training set
# This shows us how the model performs without fixing the class imbalance
# We expect it to be biased toward predicting class 0 since that is the majority

knn_unbal = KNeighborsClassifier(n_neighbors=10)
knn_unbal.fit(X_train, y_train)
y_pred_unbal = knn_unbal.predict(X_test)

# Confusion Matrixs
cm_unbal = confusion_matrix(y_test, y_pred_unbal)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_unbal, display_labels=['Not Looking', 'Looking'])
disp.plot(cmap='Blues')
plt.title('KNN (k=10) - Unbalanced Set')
plt.tight_layout()
plt.show()

print("Classification Report - Unbalanced:")
print(classification_report(y_test, y_pred_unbal, target_names=['Not Looking', 'Looking']))


### Section 6.2 - KNN with Balanced Training Set (4 pts)

In [ ]:
# Train KNN with k=10 on the BALANCED training set (after SMOTE)
# We expect better recall on class 1 since the model now sees equal examples of both

knn_bal = KNeighborsClassifier(n_neighbors=10)
knn_bal.fit(X_train_bal, y_train_bal)
y_pred_bal = knn_bal.predict(X_test)

# Confusion Matrixs
cm_bal = confusion_matrix(y_test, y_pred_bal)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_bal, display_labels=['Not Looking', 'Looking'])
disp.plot(cmap='Blues')
plt.title('KNN (k=10) - Balanced Training Set (SMOTE)')
plt.tight_layout()
plt.show()

print("Classification Report - Balanced:")
print(classification_report(y_test, y_pred_bal, target_names=['Not Looking', 'Looking']))

### Section 6.3 - Grid Search to Tune KNN Hyperparameters (3 pts)

In [ ]:
from sklearn.model_selection import GridSearchCV

# Grid search tests every combination of these parameters to find the best setup
# n_neighbors — how many nearby points to consider
# weights — uniform treats all neighbors equally, distance gives closer ones more weight
# metric — different ways to measure distance between points

param_grid = {
    'n_neighbors'   : list(range(1, 21)),
    'weights'       : ['uniform', 'distance'],
    'metric'        : ['euclidean', 'manhattan', 'minkowski']
}

knn_grid = KNeighborsClassifier()

# refit='roc_auc' means the best model is chosen based on AUC score
grid_search = GridSearchCV(
    knn_grid,
    param_grid,
    cv=5,
    scoring=['roc_auc', 'accuracy'],
    refit='roc_auc',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_bal, y_train_bal)
print("Grid search complete.")

### Section 6.4 - Print Best Parameters (2 pts)

In [ ]:
# Print the Best Parameters found by the Grid Search

print("Best Parameters found by Grid Search:")
print(grid_search.best_params_)
print(f"Best AUC Score (Cross-Validated): {grid_search.best_score_:.4f}")

### Section 6.5 - Train and Test with Best Parameters (4 pts)

In [ ]:
from sklearn.metrics import roc_auc_score

# use the best parameters from grid search to train and evaluate the model

best_knn = grid_search.best_estimator_
y_pred_best = best_knn.predict(X_test)
y_prob_best = best_knn.predict_proba(X_test)[:, 1]

cm_best = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_best, display_labels=['Not Looking', 'Looking'])
disp.plot(cmap='Blues')
plt.title('KNN - Best Parameters from Grid Search')
plt.tight_layout()
plt.show()

print("Classification Report - Best KNN")
print(classification_report(y_test, y_pred_best, target_names=['Not Looking', 'Looking']))
print(f"AUC Score: {roc_auc_score(y_test, y_prob_best):.4f}")

### Section 6.6 - PCA with Best KNN Parameters (4 pts)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

# Combine PCA dimensionality reduction with the best KNN parameters
# PCA reduces the 31 features to fewer dimensions before KNN runs
# This can improve performance by removing noise from less important features

best_params = grid_search.best_params_

knn_pca_pipeline = Pipeline([
    ('pca', PCA(n_components=21)),
    ('knn', KNeighborsClassifier(
        n_neighbors=best_params['n_neighbors'],
        weights=best_params['weights'],
        metric=best_params['metric']
    ))
])

knn_pca_pipeline.fit(X_train_bal, y_train_bal)
y_pred_pca = knn_pca_pipeline.predict(X_test)
y_prob_pca = knn_pca_pipeline.predict_proba(X_test)[:, 1]

cm_pca = confusion_matrix(y_test, y_pred_pca)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_pca, display_labels=['Not Looking', 'Looking'])
disp.plot(cmap='Blues')
plt.title('KNN with PCA - Best Parameters')
plt.tight_layout()
plt.show()

print("Classification Report - KNN with PCA:")
print(classification_report(y_test, y_pred_pca, target_names=['Not Looking', 'Looking']))
print(f"AUC Score: {roc_auc_score(y_test, y_prob_pca):.4f}")


### Section 6.7 - Discussion of All 4 KNN models (4 pts)
1) Unbalanced Training Set:
    - The model trained on the original imbalanced data performs well on class 0 (not looking) but poorly on class 1 (looking). Because class 0 makes up roughly 83% of the training data, the model is biased toward predicting the majority class, resulting in low recall for the minority class.
 2) Balanced Training Set (SMOTE):
    - After applying SMOTE to balance the training data, recall for class 1 improves noticeably. The model is no longer biased toward class 0, though this sometimes comes at the cost of slightly lower precision as more false positives appear.
3) Best Parameters (Grid Search):
    - Tuning the hyperparameters (n_neighbors, weights, and metric) via grid search improves overall AUC compared to the default k=10 model. The best parameters allow the model to draw more accurate decision boundaries for both classes.
4) KNN with PCA - Best Parameters:
    - Adding PCA before KNN reduces the feature space, which can remove noise and speed up computation.
    - Performance is comparable to the tuned KNN without PCA, though in some cases a slight drop in accuracy occurs if important information is lost during dimensionality reduction.
    - Overall, the balanced + tuned KNN model offers the best tradeoff between precision and recall for this dataset.

## Section 7 - Naive Bayes (15 pts)

### Section 7.1 - GaussianNB Model (7.5 pts)

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                              classification_report, roc_curve,
                              roc_auc_score)

# GaussianNB assumes each feature follows a normal (Gaussian) distribution
# It is a fast, simple baseline model that works well for continuous features
# We use the rebalanced training set so the model sees equal examples of both classes

gnb = GaussianNB()
gnb.fit(X_train_bal, y_train_bal)
y_pred_gnb  = gnb.predict(X_test)
y_prob_gnb  = gnb.predict_proba(X_test)[:, 1]

# Confusion matrix
cm_gnb = confusion_matrix(y_test, y_pred_gnb)
disp   = ConfusionMatrixDisplay(confusion_matrix=cm_gnb,
                                display_labels=['Not Looking', 'Looking'])
disp.plot(cmap='Blues')
plt.title('GaussianNB - Confusion Matrix')
plt.tight_layout()
plt.show()

# Classification report
print("Classification Report - GaussianNB:")
print(classification_report(y_test, y_pred_gnb,
                             target_names=['Not Looking', 'Looking']))

# ROC curve and AUC
fpr, tpr, _ = roc_curve(y_test, y_prob_gnb)
auc_gnb     = roc_auc_score(y_test, y_prob_gnb)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'GaussianNB (AUC = {auc_gnb:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - GaussianNB')
plt.legend()
plt.tight_layout()
plt.show()

print(f"AUC Score: {auc_gnb:.4f}")

# Count misclassifications
misclassified_gnb = (y_test != y_pred_gnb).sum()
print(f"Number of Misclassifications: {misclassified_gnb}")

### Section 7.1 - Observations
- GaussianNB had a rough time with this dataset. The accuracy came out to just 52%, which is barely better than randomly guessing. The main issue is that GaussianNB assumes all features follow a normal distribution, but after all the encoding and scaling we did proved the assumption does not really hold here.
- The recall for class 1 (Looking) was fairly decent at 0.76, meaning it caught a good chunk of the candidates who were job hunting. But the precision for that class was only 0.22, so for every real job-seeker it found, it wrongly flagged about 3 others. That explains the 1,291 misclassifications.
- The AUC of 0.7280 tells us the model does have some ability to separate the two classes, it just struggles with the precision side. For a use case like this where false positives waste company resources, GaussianNB alone is not a great fit.

### Section 7.2 - CategoricalNB Model (7.5 pts)

In [ ]:
from sklearn.naive_bayes import CategoricalNB
import numpy as np

# CategoricalNB assumes features are categorically distributed
# Since our data was MinMax scaled to [0, 1], we need to bin it into
# discrete integer categories before passing it to CategoricalNB
# We use 10 bins — each feature value gets assigned to one of 10 buckets

n_bins = 10
X_train_cat = np.floor(X_train_bal * n_bins).astype(int).clip(0, n_bins - 1)
X_test_cat  = np.floor(X_test      * n_bins).astype(int).clip(0, n_bins - 1)

cnb = CategoricalNB()
cnb.fit(X_train_cat, y_train_bal)
y_pred_cnb = cnb.predict(X_test_cat)
y_prob_cnb = cnb.predict_proba(X_test_cat)[:, 1]

# Confusion matrix
cm_cnb = confusion_matrix(y_test, y_pred_cnb)
disp   = ConfusionMatrixDisplay(confusion_matrix=cm_cnb,
                                display_labels=['Not Looking', 'Looking'])
disp.plot(cmap='Blues')
plt.title('CategoricalNB - Confusion Matrix')
plt.tight_layout()
plt.show()

# Classification report
print("Classification Report - CategoricalNB:")
print(classification_report(y_test, y_pred_cnb,
                             target_names=['Not Looking', 'Looking']))

# ROC curve and AUC
fpr, tpr, _ = roc_curve(y_test, y_prob_cnb)
auc_cnb     = roc_auc_score(y_test, y_prob_cnb)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'CategoricalNB (AUC = {auc_cnb:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - CategoricalNB')
plt.legend()
plt.tight_layout()
plt.show()

print(f"AUC Score: {auc_cnb:.4f}")

# Count misclassifications
misclassified_cnb = (y_test != y_pred_cnb).sum()
print(f"Number of Misclassifications: {misclassified_cnb}")

### Section 7.2 - Observations
- CategoricalNB performed noticeably better than GaussianNB once the data was binned into discrete categories. Accuracy jumped to 84% and misclassifications dropped from 1,291 down to 422, which is a huge improvement.
- Precision for class 1 came in at 0.52 and recall at 0.55, so it is more balanced compared to GaussianNB which traded precision away for recall. The AUC of 0.7545 is also slightly better.
- The binning step (10 equal-width bins) made a real difference. CategoricalNB is designed for discrete data, so forcing the scaled features into integer buckets gave it a much better representation of the data to work with. Overall CategoricalNB is the stronger of the two Naive Bayes models here.

## Section 8 - Support Vector Machine (20 pts)

### Section 8.1 - SVM Grid Search for Best Parameters (10 pts)

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# SVM finds the hyperplane that best separates the two classes
# We use grid search to tune C (regularization) and kernel type
# C controls the tradeoff between a wider margin and fewer misclassifications
# probability=True is required to get predict_proba for ROC/AUC

svm_params = {
    'C'      : [0.1, 1, 10],
    'kernel' : ['rbf', 'linear'],
    'gamma'  : ['scale', 'auto']
}

svm_grid = GridSearchCV(
    SVC(probability=True, random_state=0),
    svm_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

svm_grid.fit(X_train_bal, y_train_bal)

print("Best Parameters found by Grid Search:")
print(svm_grid.best_params_)
print(f"Best AUC Score (Cross-Validated): {svm_grid.best_score_:.4f}")

### Section 8.2 - Evaluate Best SVM on Test Set (10 pts)

In [ ]:
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                              classification_report, roc_curve, roc_auc_score)

# Use the best estimator from grid search to predict on the test set
best_svm    = svm_grid.best_estimator_
y_pred_svm  = best_svm.predict(X_test)
y_prob_svm  = best_svm.predict_proba(X_test)[:, 1]

# Confusion matrix
cm_svm = confusion_matrix(y_test, y_pred_svm)
disp   = ConfusionMatrixDisplay(confusion_matrix=cm_svm,
                                display_labels=['Not Looking', 'Looking'])
disp.plot(cmap='Blues')
plt.title('SVM - Confusion Matrix (Best Parameters)')
plt.tight_layout()
plt.show()

# Classification report
print("Classification Report - SVM:")
print(classification_report(y_test, y_pred_svm,
                             target_names=['Not Looking', 'Looking']))

# ROC curve and AUC
fpr, tpr, _ = roc_curve(y_test, y_prob_svm)
auc_svm     = roc_auc_score(y_test, y_prob_svm)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'SVM (AUC = {auc_svm:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - SVM')
plt.legend()
plt.tight_layout()
plt.show()

print(f"AUC Score: {auc_svm:.4f}")

misclassified_svm = (y_test != y_pred_svm).sum()
print(f"Number of Misclassifications: {misclassified_svm}")

### Section 8.2 - Observations
- The SVM with the best parameters (C=10, kernel=rbf, gamma=scale) landed at 83% accuracy and an AUC of 0.7313 on the test set. The cross-validated AUC during grid search was 0.8614, so there is some drop-off going from training to test, which suggests a bit of overfitting even with tuning.
- Recall for class 1 was 0.55 and precision was 0.48, which puts it in a similar range to CategoricalNB. With 463 misclassifications it sits in the middle of the pack so far.
- The rbf kernel makes sense here since the feature space after get_dummies and PCA is not linearly separable. The high C value of 10 means the model is prioritizing fewer misclassifications over a wider margin, which fits a dataset with this much class imbalance.

## Section 9 - Decision Tree (25 pts)

### Section 9.1 - Decision Tree with Unbalanced Training Set (11 pts)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                              classification_report, roc_curve, roc_auc_score)

# Decision Tree using entropy as the split criterion on the UNBALANCED training set
# We try multiple max_depth values to avoid overfitting
# entropy measures information gain — how much each split reduces uncertainty

dt_unbal = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=0)
dt_unbal.fit(X_train, y_train)
y_pred_dt_unbal = dt_unbal.predict(X_test)
y_prob_dt_unbal = dt_unbal.predict_proba(X_test)[:, 1]

# Confusion matrix
cm_dt_unbal = confusion_matrix(y_test, y_pred_dt_unbal)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_dt_unbal,
                              display_labels=['Not Looking', 'Looking'])
disp.plot(cmap='Blues')
plt.title('Decision Tree (Unbalanced) - Confusion Matrix')
plt.tight_layout()
plt.show()

# Classification report
print("Classification Report - Decision Tree (Unbalanced):")
print(classification_report(y_test, y_pred_dt_unbal,
                             target_names=['Not Looking', 'Looking']))

# ROC curve and AUC
fpr, tpr, _ = roc_curve(y_test, y_prob_dt_unbal)
auc_dt_unbal = roc_auc_score(y_test, y_prob_dt_unbal)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Decision Tree Unbalanced (AUC = {auc_dt_unbal:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Decision Tree (Unbalanced)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"AUC Score: {auc_dt_unbal:.4f}")

misclassified_dt_unbal = (y_test != y_pred_dt_unbal).sum()
print(f"Number of Misclassifications: {misclassified_dt_unbal}")

# Plot the decision tree
plt.figure(figsize=(20, 8))
plot_tree(dt_unbal, feature_names=X.columns.tolist(),
          class_names=['Not Looking', 'Looking'],
          filled=True, fontsize=8, max_depth=3)
plt.title('Decision Tree (Unbalanced) - max_depth=5')
plt.tight_layout()
plt.show()

### Section 9.1 - Observations
- The decision tree traineed on the unbalanced set hit 86% accuracy with 383 misclassifications and an AUC of 0.7334. Precision for class 1 was 0.58 and recall was 0.49, so it leans toward precision — it is more cautious about predicting job-seekers but misses about half of them.
- Using entropy as the criteron and capping max_depth at 5 kept the tree from going too deep and memorizing the training data. The tree visualization shows the early splits are on features like city_development_index and experience, which makes sense as stronger predictors of job-seeking behavior.
- The high accuracy here is partially misleadingg because the model still favors the majority class (Not Lookng) due to the imbalanced training data.

### Section 9.2 - Decision Tree with Balanced Training Set (10 pts)

In [ ]:
# Same setup as 9.1 but trained on the SMOTE-balanced training set
# We expect better recall for class 1 since the model now sees equal examples

dt_bal = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=0)
dt_bal.fit(X_train_bal, y_train_bal)
y_pred_dt_bal = dt_bal.predict(X_test)
y_prob_dt_bal = dt_bal.predict_proba(X_test)[:, 1]

# Confusion matrix
cm_dt_bal = confusion_matrix(y_test, y_pred_dt_bal)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_dt_bal,
                              display_labels=['Not Looking', 'Looking'])
disp.plot(cmap='Blues')
plt.title('Decision Tree (Balanced) - Confusion Matrix')
plt.tight_layout()
plt.show()

# Classification report
print("Classification Report - Decision Tree (Balanced):")
print(classification_report(y_test, y_pred_dt_bal,
                             target_names=['Not Looking', 'Looking']))

# ROC curve and AUC
fpr, tpr, _ = roc_curve(y_test, y_prob_dt_bal)
auc_dt_bal = roc_auc_score(y_test, y_prob_dt_bal)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Decision Tree Balanced (AUC = {auc_dt_bal:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Decision Tree (Balanced)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"AUC Score: {auc_dt_bal:.4f}")

misclassified_dt_bal = (y_test != y_pred_dt_bal).sum()
print(f"Number of Misclassifications: {misclassified_dt_bal}")

# Plot the balanced decision tree
plt.figure(figsize=(20, 8))
plot_tree(dt_bal, feature_names=X.columns.tolist(),
          class_names=['Not Looking', 'Looking'],
          filled=True, fontsize=8, max_depth=3)
plt.title('Decision Tree (Balanced) - max_depth=5')
plt.tight_layout()
plt.show()

### Section 9.3 - Discussion: Differences Between 9.1 and 9.2 (4 pts)
- The accuracy for both trees came out identical at 86% with the same 383 misclassifications, which was a bit surprising. The AUC did improve slightly from 0.7334 (unbalanced) to 0.7454 (balanced), showing the balanced model is a little better at ranking predictions.
- Looking at the classification reports more closely, the balanced tree had a marginally higher recall for class 1 (0.52 vs 0.49) at the cost of the same precision (0.58). So balancing the data did push the model to catch a few more job-seekers.
- The tree structure from 9.2 shows more branching on class 1 nodes compared to 9.1, which reflects that it learned more about the minority class after SMOTE. In practice the balanced tree is the safer choice for this problem since missing a job-seeker candidate is the more costly error for the company.

## Section 10 - Random Forest (15 pts)

### Section 10.1 - Grid Search to Tune Random Forest (8 pts)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Random Forest builds many decision trees and averages their predictions
# This reduces overfitting compared to a single tree
# We tune max_depth, min_samples_leaf, and n_estimators via grid search
# n_jobs=-1 uses all CPU cores to speed up training

rf_params = {
    'n_estimators'    : [100, 200],
    'max_depth'       : [5, 10, None],
    'min_samples_leaf': [1, 2, 4]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=0),
    rf_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train_bal, y_train_bal)

### Section 10.2 - Print Best Estimator (1 pt)

In [ ]:
# Print the full best estimator to see all parameters chosen by grid search
print("Best Estimator:")
print(rf_grid.best_estimator_)
print()
print("Best Parameters:")
print(rf_grid.best_params_)
print(f"Best AUC Score (Cross-Validated): {rf_grid.best_score_:.4f}")

### Section 10.3 - Train and Evaluate Best Random Forest (6 pts)

In [ ]:
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                              classification_report, roc_curve, roc_auc_score)

# Use the best estimator from grid search to evaluate on the test set
best_rf    = rf_grid.best_estimator_
y_pred_rf  = best_rf.predict(X_test)
y_prob_rf  = best_rf.predict_proba(X_test)[:, 1]

# Confusion matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp  = ConfusionMatrixDisplay(confusion_matrix=cm_rf,
                               display_labels=['Not Looking', 'Looking'])
disp.plot(cmap='Blues')
plt.title('Random Forest - Confusion Matrix (Best Parameters)')
plt.tight_layout()
plt.show()

# Classification report
print("Classification Report - Random Forest:")
print(classification_report(y_test, y_pred_rf,
                             target_names=['Not Looking', 'Looking']))

# ROC curve and AUC
fpr, tpr, _ = roc_curve(y_test, y_prob_rf)
auc_rf      = roc_auc_score(y_test, y_prob_rf)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Random Forest (AUC = {auc_rf:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Random Forest')
plt.legend()
plt.tight_layout()
plt.show()

print(f"AUC Score: {auc_rf:.4f}")

misclassified_rf = (y_test != y_pred_rf).sum()
print(f"Number of Misclassifications: {misclassified_rf}")

### Section 10.3 - Observations
- Random Forest with the best parameters (n_estimators=200, max_depth=None, min_samples_leaf=1) achieved 84% accuracy, AUC of 0.7448, and 429 misclassifications. The cross-validated AUC during grid search was 0.9639, which is the highest we have seen across all models, though the test AUC is more modest.
- The gap between CV AUC (0.9639) and test AUC (0.7448) could suggests overfitting. Allowing max_depth=None means individual trees can grow very deep, and while averaging 200 of them reduces variance, it does not eliminate variance entirely.
- Precision for class 1 was 0.52 and recall was 0.47. Compared to the decision tree, Random Forest does not offer a increase in performance on the test set, but the ensemble approach not only makes it more stable but also less sensitive to noise in the training data.

## Section 11 - Boosting Algorithms (10 pts)

### Section 11.1 - AdaBoost Classifier (5 pts)

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                              classification_report, roc_curve, roc_auc_score)

# AdaBoost (Adaptive Boosting) trains weak learners sequentially
# Each new tree focuses more on the examples the previous tree got wrong
# n_estimators is the number of weak learners to chain together
# learning_rate controls how much each tree contributes to the final prediction

ada = AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=0)
ada.fit(X_train_bal, y_train_bal)
y_pred_ada = ada.predict(X_test)
y_prob_ada = ada.predict_proba(X_test)[:, 1]

# Confusion matrix
cm_ada = confusion_matrix(y_test, y_pred_ada)
disp   = ConfusionMatrixDisplay(confusion_matrix=cm_ada,
                                display_labels=['Not Looking', 'Looking'])
disp.plot(cmap='Blues')
plt.title('AdaBoost - Confusion Matrix')
plt.tight_layout()
plt.show()

# Classification report
print("Classification Report - AdaBoost:")
print(classification_report(y_test, y_pred_ada,
                             target_names=['Not Looking', 'Looking']))

# ROC curve and AUC
fpr, tpr, _ = roc_curve(y_test, y_prob_ada)
auc_ada     = roc_auc_score(y_test, y_prob_ada)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'AdaBoost (AUC = {auc_ada:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - AdaBoost')
plt.legend()
plt.tight_layout()
plt.show()

print(f"AUC Score: {auc_ada:.4f}")

misclassified_ada = (y_test != y_pred_ada).sum()
print(f"Number of Misclassifications: {misclassified_ada}")

### Section 11.1 - Observations
- AdaBoost reached 85% accuracy with 398 misclassifications and an AUC of 0.7496. Precision for class 1 was 0.55 and recall was 0.54, making it one of the more balanced models we have tested for the minority class.
- The sequential boosting process helped here — each round focused more attention on the candidates that were previously misclassified, which is exactly what we need for a dataset where class 1 is hard to identify. The learning rate of 0.1 kept individual trees from overcorrecting.
- Compared to a single decision tree, AdaBoost improved the class 1 F1-score from 0.53 to 0.55 while keeping similar overall accuracy. It is a solid middle-ground model.

### Section 11.2 - Gradient Boosting Classifier (5 pts)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Gradient Boosting also trains trees sequentially like AdaBoost
# but it fits each new tree to the residual errors of the previous model
# rather than reweighting misclassified samples
# subsample < 1.0 adds randomness (stochastic gradient boosting) which helps generalization

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                 max_depth=3, subsample=0.8, random_state=0)
gb.fit(X_train_bal, y_train_bal)
y_pred_gb = gb.predict(X_test)
y_prob_gb = gb.predict_proba(X_test)[:, 1]

# Confusion matrix
cm_gb = confusion_matrix(y_test, y_pred_gb)
disp  = ConfusionMatrixDisplay(confusion_matrix=cm_gb,
                               display_labels=['Not Looking', 'Looking'])
disp.plot(cmap='Blues')
plt.title('Gradient Boosting - Confusion Matrix')
plt.tight_layout()
plt.show()

# Classification report
print("Classification Report - Gradient Boosting:")
print(classification_report(y_test, y_pred_gb,
                             target_names=['Not Looking', 'Looking']))

# ROC curve and AUC
fpr, tpr, _ = roc_curve(y_test, y_prob_gb)
auc_gb      = roc_auc_score(y_test, y_prob_gb)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Gradient Boosting (AUC = {auc_gb:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Gradient Boosting')
plt.legend()
plt.tight_layout()
plt.show()

print(f"AUC Score: {auc_gb:.4f}")

misclassified_gb = (y_test != y_pred_gb).sum()
print(f"Number of Misclassifications: {misclassified_gb}")

### Section 11.2 - Observations
- Gradient Boosting was the strongest overall performer with 86% accuracy, 388 misclassifications, and the highest test AUC of 0.7692 across all models tested.
- Precision for class 1 was 0.57 and recall was 0.52, which is the best precision we have seen for the Looking class outside of the unbalanced Decision Tree. The residual-fitting approach works well here since each tree corrects the specific errors of the previous one rather than just reweighting them like AdaBoost.
- The subsample=0.8 setting added a stochastic element that helped prevent overfitting. Gradient Boosting taking the top AUC spot makes it the most reliable model for this problem.

## Section 12 - Final Discussion (10 pts)

### Section 12.1 - Which Model is Most Suitable and Future Work
- Looking across all the models Gradient Boosting stands out as the most suitable candidate for this scenario. Gradient Boosting had the highest test AUC (0.7692), strong overall accuracy (86%), and the best balance between precision and recall for class 1. In this example for a company trying to identify which candidates are likely to leave, AUC is the most meaningful metric since it measures how well the model ranks job-seekers above non-job seekers.
- CategoricalNB was a surprisingly strong baseline with only 422 misclassifications and an AUC of 0.7545, despite given how simple the model is. It would be a good lightweight option if compute resources were limited.
- Honestly one of the biggest suprise was the result of GaussianNB, who was clearly the weakest model. The 1,291 misclassifications and 52% accuracy show it is not well suited for this type of feature space. Models that make fewer distributional assumptions, like tree-based methods, consistently performed better here.
- For future work, a few things could push performance further.
    - First, tuning the classification threshold rather than defaulting to 0.5 could improve recall for class 1 since catching job seekers is more valuable than avoiding false positives.
    - Second, the feature importance from Random Forest or Gradient Boosting could guide targeted feature engineering for example, creating interaction terms between city_development_index and experience.
    - Third, trying XGBoost or LightGBM could squeeze out additional performance since they handle imbalanced data better than sklearn's GradientBoostingClassifier out of the box.   
- Finally, collecting more data for class 1 candidates over time would reduce the dependence on SMOTE, which generates synthetic samples rather than real observations.